In [1]:
import os
import json
import pickle
import numpy as np

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PATHS = {
    "chunks": DATA_PROCESSED_DIR / "chunks.json",
    "faiss_index": DATA_PROCESSED_DIR / "faiss_index.bin",
    "bm25_index": DATA_PROCESSED_DIR / "bm25_index.pkl",
    "embeddings": DATA_PROCESSED_DIR / "embeddings.npy"
}

if PATHS["chunks"].exists():
    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f"Loaded chunks.json successfully!")
    print(f"Total chunks: {len(chunks):,}")
else:
    raise FileNotFoundError(f"Chunks file not found: {PATHS['chunks']}")

Loaded chunks.json successfully!
Total chunks: 74,844


## FAISS + BM25

In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

def build_dense_faiss_index():
    print("=== BUILDING DENSE FAISS VECTOR INDEX ===")

    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    texts = [c["text"] for c in chunks]

    print(f"Total texts: {len(texts)}")
    print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")

    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print("Generating embeddings...")

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    embeddings = embeddings.astype("float32")

    print("Embedding shape:", embeddings.shape)

    np.save(PATHS["embeddings"], embeddings)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    faiss.write_index(index, str(PATHS["faiss_index"]))

    print("\n=== DENSE INDEX COMPLETE ===")
    print("Total vectors:", index.ntotal)
    print("Vector dimension:", dimension)
    print("FAISS index saved:", PATHS["faiss_index"])
    print("Embeddings saved:", PATHS["embeddings"])

build_dense_faiss_index()

=== BUILDING DENSE FAISS VECTOR INDEX ===
Total texts: 74844
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/2339 [00:00<?, ?it/s]

Embedding shape: (74844, 384)

=== DENSE INDEX COMPLETE ===
Total vectors: 74844
Vector dimension: 384
FAISS index saved: E:\Rasengan\College-Knowledge-Search\data\processed\faiss_index.bin
Embeddings saved: E:\Rasengan\College-Knowledge-Search\data\processed\embeddings.npy


#### BM25 - BM25Okapi

In [3]:
import pickle
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b[\w-]+\b", text.lower())

def build_sparse_bm25_index():
    print("=== BUILDING SPARSE BM25 KEYWORD INDEX ===")

    with open(PATHS["chunks"], "r", encoding="utf-8") as f:
        chunks = json.load(f)

    corpus = [tokenize(c["text"]) for c in chunks]

    print(f"Total documents: {len(corpus)}")

    print("Building BM25 index...")
    bm25 = BM25Okapi(corpus)

    with open(PATHS["bm25_index"], "wb") as f:
        pickle.dump(bm25, f)

    print("\n=== SPARSE INDEX COMPLETE ===")
    print(f"Total documents indexed: {len(corpus)}")
    print(f"Saved BM25 index to: {PATHS['bm25_index']}")

build_sparse_bm25_index()

=== BUILDING SPARSE BM25 KEYWORD INDEX ===
Total documents: 74844
Building BM25 index...

=== SPARSE INDEX COMPLETE ===
Total documents indexed: 74844
Saved BM25 index to: E:\Rasengan\College-Knowledge-Search\data\processed\bm25_index.pkl


In [4]:
faiss_exists = os.path.exists(PATHS["faiss_index"])
bm25_exists = os.path.exists(PATHS["bm25_index"])

print(f"FAISS Dense Vector Index: {' READY' if faiss_exists else ' MISSING'}")
print(f"BM25 Sparse Keyword Index: {' READY' if bm25_exists else ' MISSING'}")

if faiss_exists and bm25_exists:
    faiss_size = os.path.getsize(PATHS["faiss_index"]) / (1024 * 1024)
    bm25_size = os.path.getsize(PATHS["bm25_index"]) / (1024 * 1024)
    print(f"\nFAISS File Size: {faiss_size:.2f} MB")
    print(f"BM25 File Size: {bm25_size:.2f} MB")

FAISS Dense Vector Index:  READY
BM25 Sparse Keyword Index:  READY

FAISS File Size: 109.63 MB
BM25 File Size: 30.28 MB
